In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("benchmark_script_results")
SWEEP = "warm-up-1-step__measurement-10-steps"

SIZES = ["small", "medium", "large", "xl", "10B"]
FUNCS = ["forward", "forward_backward", "forward_backward_optimizer"]

In [ ]:
paths = sorted((RESULTS_DIR / SWEEP).glob("*.json"))
df = pd.DataFrame([json.loads(p.read_text()) for p in paths])
df["timestamp"] = pd.to_datetime(df["timestamp"])
df[["model_size", "func_to_time", "status", "mean_sec", "std_sec", "git_commit", "timestamp"]]
df.head()

In [ ]:
assert len(df) > 0, f"no records in {RESULTS_DIR / SWEEP}"
print("commits:", df["git_commit"].unique())
print(df["status"].value_counts())

df = (
    df.sort_values("timestamp")
      .drop_duplicates(subset=["model_size", "func_to_time"], keep="last")
)

In [ ]:
def make_table(value_col: str) -> pd.DataFrame:
    values = (
        df.pivot(index="func_to_time", columns="model_size", values=value_col)
          .reindex(index=FUNCS, columns=SIZES)
    )
    status = (
        df.pivot(index="func_to_time", columns="model_size", values="status")
          .reindex(index=FUNCS, columns=SIZES)
    )
    return values.map(lambda x: f"{x:.4g}").where(status == "success", status.fillna("TBD"))

mean_table = make_table("mean_sec")
std_table = make_table("std_sec")

In [ ]:
print("Mean (seconds)")
display(mean_table)
print("Std (seconds)")
display(std_table)